In [1]:
import os
%pwd

'c:\\Users\\HP\\Desktop\\coding\\kidney-disease-classification\\research'

In [2]:
os.chdir('../')
%pwd

'c:\\Users\\HP\\Desktop\\coding\\kidney-disease-classification'

In [3]:

from dataclasses import dataclass
from pathlib import Path

In [4]:
@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [5]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

In [6]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )


        return data_ingestion_config

In [7]:
import os
import zipfile
import gdown
from cnnClassifier import logger
from cnnClassifier.utils.common import get_size

In [8]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    
    def download_file(self)-> str:
        '''
        Fetch data from the url
        '''

        try: 
            dataset_url = self.config.source_URL
            zip_download_dir = self.config.local_data_file
            os.makedirs("artifacts/data_ingestion", exist_ok=True)
            logger.info(f"Downloading data from {dataset_url} into file {zip_download_dir}")

            file_id = dataset_url.split("/")[-2]
            prefix = 'https://drive.google.com/uc?/export=download&id='
            gdown.download(prefix+file_id,zip_download_dir)

            logger.info(f"Downloaded data from {dataset_url} into file {zip_download_dir}")

        except Exception as e:
            raise e
        
    

    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [9]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-10-17 14:41:59,761: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-10-17 14:41:59,766: INFO: common: yaml file: params.yaml loaded successfully]
[2026-10-17 14:41:59,771: INFO: common: created directory at: artifacts]
[2026-10-17 14:41:59,779: INFO: common: created directory at: artifacts/data_ingestion]
[2026-10-17 14:41:59,783: INFO: 986331348: Downloading data from https://drive.google.com/file/d/18RmxqG-RItRmQNV7JvA__ik-eF6EGRPt/view?usp=sharing into file artifacts/data_ingestion/data.zip]


Downloading...
From (original): https://drive.google.com/uc?/export=download&id=18RmxqG-RItRmQNV7JvA__ik-eF6EGRPt
From (redirected): https://drive.google.com/uc?%2Fexport=download&id=18RmxqG-RItRmQNV7JvA__ik-eF6EGRPt&confirm=t&uuid=2261d3ce-2925-4951-aaa7-42e3e58c2817
To: c:\Users\HP\Desktop\coding\kidney-disease-classification\artifacts\data_ingestion\data.zip
100%|██████████| 1.63G/1.63G [18:23<00:00, 1.47MB/s]

[2026-10-17 15:00:29,123: INFO: 986331348: Downloaded data from https://drive.google.com/file/d/18RmxqG-RItRmQNV7JvA__ik-eF6EGRPt/view?usp=sharing into file artifacts/data_ingestion/data.zip]
